# 01. 평가셋 설계 (Eval-first)

> 재설계 1단계 · 전체 계획은 [`plan.md`](plan.md) 참고
> 산출물: `eval/eval_set.json`(평가셋), `eval/gold_texts.json`(정답 근거 원문 캐시)

## 이 노트북이 결정하는 것

| 결정 | 요약 |
|---|---|
| 무엇으로 평가하는가 | 12개 핵심 질문 + 변형 질문, 질문별 **정답 근거(조문·판례·용어)**, 필수 포함 요소, 금지 표현 |
| 정답 근거는 믿을 만한가 | 법령 API로 **원문을 직접 받아** 가설을 검증하고, 틀린 가설은 수정 |
| 어떤 지표를 쓰는가 | 검색: Recall@k, MRR / 생성: 필수 요소 포함률, 확답 회피, 범위 밖 처리 정확도, 근거 충실도 |

## 1. 왜 평가셋부터 만드는가

기존 파이프라인은 청킹·임베딩·LLM·게이트 노드가 한꺼번에 바뀌어서, 답변이 나빠도 **어느 부분의 문제인지 구분할 수 없었다.**
이번에는 평가 기준을 먼저 고정하고 한 번에 한 가지만 바꾸어 비교한다.

**참고한 방법론**

- **RAGAS** (Es et al., 2023): RAG 평가를 *검색 품질*(context precision/recall)과 *생성 품질*(faithfulness, answer relevancy)로 나눈다.
  → 이 프로젝트도 4단계(검색)와 6단계(LLM)를 따로 평가한다.
- **LegalBench-RAG** (Pipitone & Alami, 2024): 법률 RAG에서는 "답이 그럴듯한가"보다 **정확한 근거 조각을 찾았는가**가 핵심이라고 보고, 정답을 문서 단위가 아니라 *근거 구간* 단위로 둔다.
  → 여기서는 정답 근거를 **조문 단위**(예: 민법 제636조)와 **판례 사건번호 단위**로 정의한다.

**이 프로젝트에 맞춘 추가 기준**

- **확답 회피**: 변호사법·공인중개사법 문제를 피하려면 "~할 수 있습니다/없습니다"처럼 결론을 단정하면 안 된다. 따라서 질문마다 `must_not`(금지 표현)을 둔다.
- **범위 밖 처리**: Q3(귀신), Q8(공인중개사 시험 준비)은 근거 없이 답하지 않고 정중하게 범위 밖임을 알리는지 본다.

## 2. 질문 분류 체계

12개 질문을 **어떤 종류의 근거가 필요한가** 기준으로 나눈다. 이 분류가 2단계(데이터 소스 선정)의 입력이 된다.

| category | 의미 | 필요한 근거 | 질문 |
|---|---|---|---|
| `term` | 용어 뜻 | 법령용어 사전 + 정의 조문 | Q6 |
| `procedure` | 행정 절차·기한 | 법률 + 시행령(세부 기준) | Q12 |
| `dispute` | 분쟁·갈등 대응 | 민법 조문 + 판례 | Q1, Q2, Q7, Q9, Q11 |
| `special_terms` | 특약 작성 | 민법 조문 + (특약 사례) | Q4, Q5 |
| `policy` | 지원 정책 | **법령 API 밖의 외부 공공 소스** | Q10 |
| `out_of_scope` | 서비스 범위 밖 | 없음 (정중히 거절) | Q3, Q8 |
| `time` | 특정 시점의 법 적용 | **그 시점에 시행 중이던 조문 판본** + 시행일 | T01, T02 (v0.3 추가) |

In [1]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import dotenv_values

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
  ROOT = ROOT.parent
HERE = ROOT / "notebooks" / "redesign"
EVAL_PATH = HERE / "eval" / "eval_set.json"
GOLD_CACHE_PATH = HERE / "eval" / "gold_texts.json"

OC = dotenv_values(ROOT / ".env").get("OC")
assert OC, ".env에 법령 API 인증키 OC가 필요합니다."

pd.set_option("display.max_colwidth", 120)

## 3. 평가셋 불러오기

평가셋 원본은 `eval/eval_set.json`에 두고, 노트북은 읽기와 검증만 한다. 평가셋을 고치면 버전(`version`)을 올린다.

- `question`: 팀원이 작성한 원문 질문 (오타 포함 그대로 유지 — 실제 사용자 입력과 비슷하게)
- `variants`: 같은 의도를 다른 말투로 바꾼 질문. 12문항만으로는 통계적으로 약하기 때문에 검색 평가에서 표본을 늘리는 용도
- `gold_sources`: 정답 근거 가설 (아래 4절에서 API로 검증)
- `must_include` / `must_not`: 생성 평가용 체크리스트

In [2]:
eval_set = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
items = eval_set["items"]

summary = pd.DataFrame(
  [
    {
      "id": it["id"],
      "persona": it["persona"],
      "category": it["category"],
      "expected": it["expected_behavior"],
      "n_variants": len(it["variants"]),
      "n_gold": len(it["gold_sources"]),
      "question": it["question"][:50],
    }
    for it in items
  ]
)
n_queries = len(summary) + summary["n_variants"].sum()
print(f"평가셋 v{eval_set['version']} — 핵심 {len(summary)}문항, 변형 포함 총 {n_queries}개 질의")
summary

평가셋 v0.3.0 — 핵심 14문항, 변형 포함 총 45개 질의


,id,persona,category,expected,n_variants,n_gold,question
0,Q01,석원,dispute,answer,3,2,월세 계약은 1년을 했는데 중간에 나가야할 일이 생겼는데 어떻게 해야할까?
1,Q02,정희,dispute,answer,3,3,"내가 지금 전세금 3억정도의 집에 살고있는데, 전세 계약 만료일이 1년 정도 남았단 말이야"
2,Q03,정희,out_of_scope,decline_gracefully,1,0,나 여기 계약했는데 귀신나와 이거 해지 사유 되나?
3,Q04,세형,special_terms,answer,3,2,나는 자금이 부족한 상황이야. 그래서 전세자금 대출을 기반으로 계약을 진행하려고 해. 그러
4,Q05,정현,special_terms,answer,3,2,내가 키우고 있는 고양이가 있는데 임대인이 특약을 넣자고 하네. 최대한 부담되지 않게 작성
5,Q06,아라,term,answer,2,2,임대차 계약서 차임 뜻이 뭐야?
6,Q07,정현,dispute,answer,2,2,윗집이 밤마다 소음이 심한데 해결 방법 있을까?
7,Q08,정현,out_of_scope,decline_gracefully,1,0,공인중개사가 되려면 어떻게 준비해야돼?
8,Q09,아라,dispute,answer,3,3,"다른 사람이 집을 계약할까봐 보증금의 10프로는 먼저 지불했는데, 내가 계약을 못하게 되었"
9,Q10,정희,policy,answer_with_external,2,1,혹시 자취생을 위한 주거비 지원 정책 좀 아는 거 있니?


In [3]:
summary.groupby("category").agg(n=("id", "count"), ids=("id", ", ".join))

,n,ids
category,,
dispute,5,"Q01, Q02, Q07, Q09, Q11"
out_of_scope,2,"Q03, Q08"
policy,1,Q10
procedure,1,Q12
special_terms,2,"Q04, Q05"
term,1,Q06
time,2,"T01, T02"


## 4. 정답 근거 검증 (법령 API)

정답 근거는 사람이 기억에 의존해 적은 **가설**이다. 조문 번호가 틀리거나 개정되었을 수 있으므로,
국가법령정보 공동활용 API로 **원문을 직접 받아 확인**한다.

**구현 메모**
- 서비스 코드(`app.clients`)를 import하면 `app/__init__.py`가 FastAPI 라우터와 OCR·torch까지 불러와 느리다.
  평가 노트북은 가벼워야 하므로 `requests`로 직접 호출한다.
- 조문 조회는 `lawService.do?target=law&ID=<법령ID>&JO=<6자리>`를 쓴다. `JO`는 조문번호 4자리와 가지번호 2자리다(제6조의2 → `000602`).
- 검색 결과가 1건이면 리스트가 아니라 dict로 오므로 `as_list`로 통일한다.

In [4]:
SEARCH_URL = "https://www.law.go.kr/DRF/lawSearch.do"
SERVICE_URL = "https://www.law.go.kr/DRF/lawService.do"
_session = requests.Session()


def law_api(url: str, **params) -> dict:
  resp = _session.get(url, params={"OC": OC, "type": "JSON", **params}, timeout=30)
  resp.raise_for_status()
  time.sleep(0.3)  # 공용 API 부하 방지
  return resp.json()


def as_list(x) -> list:
  if x is None:
    return []
  return x if isinstance(x, list) else [x]


def _join(x) -> str:
  if isinstance(x, list):
    return "\n".join(_join(v) for v in x)
  return str(x or "").strip()


def article_text(unit: dict) -> str:
  """조문단위 dict → 사람이 읽을 수 있는 조문 텍스트 (항·호 포함)."""
  lines = [_join(unit.get("조문내용"))]
  for hang in as_list(unit.get("항")):
    lines.append(_join(hang.get("항내용")))
    for ho in as_list(hang.get("호")):
      lines.append("  " + _join(ho.get("호내용")))
  return "\n".join(line for line in lines if line)


_law_id_cache: dict[str, str | None] = {}


def find_law_id(name: str) -> str | None:
  """법령명과 정확히 일치하는 현행 법령의 법령ID."""
  if name not in _law_id_cache:
    res = law_api(SEARCH_URL, target="law", query=name, display=100)["LawSearch"]
    _law_id_cache[name] = next(
      (row["법령ID"] for row in as_list(res.get("law")) if row["법령명한글"] == name),
      None,
    )
  return _law_id_cache[name]


def article_label(unit: dict) -> str:
  """조문번호·가지번호 → '제6조의2' 형식."""
  label = f"제{unit['조문번호']}조"
  if unit.get("조문가지번호"):
    label += f"의{unit['조문가지번호']}"
  return label


def fetch_articles(law_id: str, jo: str | None = None) -> tuple[dict, list[dict]]:
  """(기본정보, 조문 리스트). jo가 없으면 법령 전체 조문."""
  params = {"target": "law", "ID": law_id}
  if jo:
    params["JO"] = jo
  body = law_api(SERVICE_URL, **params)["법령"]
  units = [u for u in as_list(body["조문"]["조문단위"]) if u.get("조문여부") == "조문"]
  return body["기본정보"], units

### 4-1. 조문 근거

`jo`가 있으면 해당 조문을 받아 조문 제목을 확인하고, `jo`가 없는 항목(시행령·규칙)은 법령 전체에서 `title_keyword`가 들어간 조문을 찾는다.

In [5]:
TITLE_KEYWORDS = {
  # jo를 미리 알 수 없는 하위 법령: 조문제목 키워드로 찾는다
  "공동주택 층간소음의 범위와 기준에 관한 규칙": "층간소음",
  "부동산 거래신고 등에 관한 법률 시행령": "임대차 계약의 신고",
}

gold_texts: dict[str, dict] = {}
law_rows = []

for it in items:
  for src in it["gold_sources"]:
    if src["type"] != "law":
      continue
    law_id = find_law_id(src["law_name"])
    row = {"id": it["id"], "law": src["law_name"], "hypothesis": src["article"], "law_id": law_id}
    if law_id is None:
      law_rows.append({**row, "status": "법령 없음"})
      continue

    info, units = fetch_articles(law_id, src.get("jo"))
    if not src.get("jo"):
      kw = TITLE_KEYWORDS[src["law_name"]]
      units = [u for u in units if kw in _join(u.get("조문제목"))]

    for u in units:
      key = f"{src['law_name']}|{article_label(u)}"
      gold_texts[key] = {
        "law_name": src["law_name"],
        "law_id": law_id,
        "시행일자": info.get("시행일자"),
        "조문제목": _join(u.get("조문제목")),
        "text": article_text(u),
      }
    law_rows.append(
      {
        **row,
        "시행일자": info.get("시행일자"),
        "found": [f"{article_label(u)}({_join(u.get('조문제목'))})" for u in units],
        "status": "확인" if units else "조문 없음",
      }
    )

law_df = pd.DataFrame(law_rows)
law_df

,id,law,hypothesis,law_id,시행일자,found,status
0,Q01,민법,제636조,001706,20260317,[제636조(기간의 약정있는 임대차의 해지통고)],확인
1,Q01,민법,제635조,001706,20260317,[제635조(기간의 약정없는 임대차의 해지통고)],확인
2,Q02,민법,제544조,001706,20260317,[제544조(이행지체와 해제)],확인
3,Q02,민법,제303조,001706,20260317,[제303조(전세권의 내용)],확인
4,Q02,민법,제390조,001706,20260317,[제390조(채무불이행과 손해배상)],확인
5,Q04,민법,제147조,001706,20260317,[제147조(조건성취의 효과)],확인
6,Q04,민법,제565조,001706,20260317,[제565조(해약금)],확인
7,Q05,민법,제654조,001706,20260317,[제654조(준용규정)],확인
8,Q05,민법,제615조,001706,20260317,[제615조(차주의 원상회복의무와 철거권)],확인
9,Q06,민법,제618조,001706,20260317,[제618조(임대차의 의의)],확인


In [6]:
# 검증된 조문 원문 일부 확인
for key in list(gold_texts)[:4]:
  g = gold_texts[key]
  print(f"■ {g['law_name']} — {g['조문제목']} (시행 {g['시행일자']})")
  print(g["text"][:300], "\n")

■ 민법 — 기간의 약정있는 임대차의 해지통고 (시행 20260317)
제636조(기간의 약정있는 임대차의 해지통고) 임대차기간의 약정이 있는 경우에도 당사자일방 또는 쌍방이 그 기간내에 해지할 권리를 보류한 때에는 전조의 규정을 준용한다. 

■ 민법 — 기간의 약정없는 임대차의 해지통고 (시행 20260317)
제635조(기간의 약정없는 임대차의 해지통고)
①임대차기간의 약정이 없는 때에는 당사자는 언제든지 계약해지의 통고를 할 수 있다.
②상대방이 전항의 통고를 받은 날로부터 다음 각호의 기간이 경과하면 해지의 효력이 생긴다.
  1. 토지, 건물 기타 공작물에 대하여는 임대인이 해지를 통고한 경우에는 6월, 임차인이 해지를 통고한 경우에는 1월
  2. 동산에 대하여는 5일 

■ 민법 — 이행지체와 해제 (시행 20260317)
제544조(이행지체와 해제) 당사자 일방이 그 채무를 이행하지 아니하는 때에는 상대방은 상당한 기간을 정하여 그 이행을 최고하고 그 기간내에 이행하지 아니한 때에는 계약을 해제할 수 있다. 그러나 채무자가 미리 이행하지 아니할 의사를 표시한 경우에는 최고를 요하지 아니한다. 

■ 민법 — 전세권의 내용 (시행 20260317)
제303조(전세권의 내용)
①전세권자는 전세금을 지급하고 타인의 부동산을 점유하여 그 부동산의 용도에 좇아 사용ㆍ수익하며, 그 부동산 전부에 대하여 후순위권리자 기타 채권자보다 전세금의 우선변제를 받을 권리가 있다. <개정 1984.4.10>
②농경지는 전세권의 목적으로 하지 못한다. 



### 4-2. 판례 근거

판례는 조문처럼 번호를 미리 알 수 없으므로 **검색어로 찾은 뒤 사람이 확인**한다.
판시사항·참조조문을 함께 보면, 가설로 적은 조문이 맞는지도 교차 확인할 수 있다.

In [7]:
prec_rows = []
for it in items:
  for src in it["gold_sources"]:
    if src["type"] != "prec":
      continue
    res = law_api(SEARCH_URL, target="prec", query=src["query"], display=20)["PrecSearch"]
    for p in as_list(res.get("prec")):
      detail = law_api(SERVICE_URL, target="prec", ID=p["판례일련번호"])["PrecService"]
      clean = lambda s: re.sub(r"<br\s*/?>", " ", s or "").strip()
      prec_rows.append(
        {
          "id": it["id"],
          "사건번호": p["사건번호"],
          "gold_일치": p["사건번호"] == src.get("case_no"),
          "사건명": p["사건명"],
          "선고일자": p["선고일자"],
          "판시사항": clean(detail.get("판시사항")),
          "참조조문": clean(detail.get("참조조문")),
          "판결요지_길이": len(clean(detail.get("판결요지"))),
        }
      )
      gold_texts[f"판례|{p['사건번호']}"] = {
        "사건번호": p["사건번호"],
        "사건명": p["사건명"],
        "판시사항": clean(detail.get("판시사항")),
        "참조조문": clean(detail.get("참조조문")),
      }

prec_df = pd.DataFrame(prec_rows)
prec_df

,id,사건번호,gold_일치,사건명,선고일자,판시사항,참조조문,판결요지_길이
0,Q09,2022다247187,True,임차보증가계약금반환,2022.09.29,가계약금에 관하여 해약금 약정이 있었다고 인정하기 위한 요건,"민법 제105조, 제568조",0


### 4-3. 법령용어 근거

법령용어 검색(`target=lstrm`)은 같은 용어가 **여러 사전**에 걸쳐 나온다.
사전별 정의를 모두 받아 보고, 사회초년생에게 설명할 때 쓸 수 있는 정의가 어느 사전에 있는지 확인한다.

In [8]:
term_rows = []
for it in items:
  for src in it["gold_sources"]:
    if src["type"] != "lstrm":
      continue
    res = law_api(SEARCH_URL, target="lstrm", query=src["term"], display=50)["LsTrmSearch"]
    for t in as_list(res.get("lstrm")):
      if t["법령용어명"] != src["term"]:
        continue
      d = law_api(SERVICE_URL, target="lstrm", trmSeqs=t["법령용어ID"])["LsTrmService"]
      term_rows.append(
        {
          "id": it["id"],
          "용어": d.get("법령용어명_한글"),
          "사전": d.get("법령용어코드명"),
          "정의": _join(d.get("법령용어정의"))[:200],
        }
      )

term_df = pd.DataFrame(term_rows)
for r in term_rows:
  gold_texts[f"용어|{r['용어']}|{r['사전']}"] = r
term_df

,id,용어,사전,정의
0,Q06,차임,법령한영사전,rent


In [9]:
GOLD_CACHE_PATH.write_text(json.dumps(gold_texts, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"정답 근거 원문 {len(gold_texts)}건 저장 → {GOLD_CACHE_PATH.relative_to(ROOT)}")

정답 근거 원문 21건 저장 → notebooks\redesign\eval\gold_texts.json


## 5. 확답 회피 기준

"확답"은 **개별 사안에 대해 법적 결론을 단정하는 표현**으로 정의한다.
일반적인 법 규정 설명("민법 제636조는 ~라고 정한다")은 허용하고, 사용자의 사안에 결론을 내리는 것("당신은 계약을 해지할 수 있습니다")은 금지한다.

평가는 두 층으로 한다.

1. **규칙 기반 1차 필터** (아래 정규식): 빠르고 재현 가능하지만 놓치는 경우가 있다.
2. **LLM 평가자 2차 판정** (6단계에서 구현): 아래 루브릭으로 판정하고, 사람이 채점한 샘플과 일치율을 확인한다.
   (LLM 평가자는 자기 모델 답변을 후하게 평가하는 편향이 보고되어 있어 — Zheng et al., 2023, *Judging LLM-as-a-Judge* — 사람 채점과 병행한다.)

In [10]:
DEFINITIVE_PATTERNS = [
  r"(반드시|무조건|확실히|100\s*%)",
  r"(해지|해제|파기|반환|청구)(할|받을)\s*수\s*(있습니다|없습니다)\s*[.!]",
  r"(승소|이길\s*수)",
  r"(성립합니다|성립하지 않습니다)",
]
HEDGE_PATTERNS = [r"수\s*있을\s*수", r"경우에\s*따라", r"전문가", r"상담", r"달라질\s*수"]


def rule_check(answer: str) -> dict:
  hits = [p for p in DEFINITIVE_PATTERNS if re.search(p, answer)]
  hedges = [p for p in HEDGE_PATTERNS if re.search(p, answer)]
  return {"definitive_hits": hits, "has_hedge": bool(hedges), "pass": not hits and bool(hedges)}


# 규칙이 의도대로 동작하는지 예시로 확인
samples = {
  "단정(실패해야 함)": "이 경우 계약을 해지할 수 있습니다. 반드시 내용증명을 보내세요.",
  "조건부(통과해야 함)": "특약 내용에 따라 해지가 가능할 수도 있으니, 계약서를 확인하고 전문가와 상담해 보시길 권합니다.",
  "일반 규정 설명(통과해야 함)": "민법 제636조는 기간 약정이 있어도 해지권을 보류한 경우 해지통고를 할 수 있다고 정합니다. 구체적인 판단은 상황에 따라 달라질 수 있습니다.",
}
pd.DataFrame({k: rule_check(v) for k, v in samples.items()}).T

,definitive_hits,has_hedge,pass
단정(실패해야 함),"[(반드시|무조건|확실히|100\s*%), (해지|해제|파기|반환|청구)(할|받을)\s*수\s*(있습니다|없습니다)\s*[.!]]",False,False
조건부(통과해야 함),[],True,True
일반 규정 설명(통과해야 함),[],True,True


**LLM 평가자 루브릭 (초안, 6단계에서 사용)**

| 항목 | 척도 | 기준 |
|---|---|---|
| 확답 회피 | pass / fail | 사용자 사안에 법적 결론을 단정하면 fail |
| 필수 요소 포함률 | 0~1 | `must_include` 항목 중 답변이 다룬 비율 |
| 금지 표현 | pass / fail | `must_not` 항목 중 하나라도 있으면 fail |
| 근거 충실도 | 1~5 | 답변의 법적 주장이 인용한 근거 문서로 뒷받침되는 정도 (RAGAS faithfulness 개념) |
| 범위 밖 처리 | pass / fail | `expected_behavior=decline_gracefully`일 때 근거 없는 답을 만들지 않으면 pass |

## 6. 지표 정의

| 단계 | 지표 | 정의 | 대상 |
|---|---|---|---|
| 검색 (4단계) | **Recall@k** | top-k 안에 정답 근거가 하나라도 있는 질의 비율 | 핵심 + 변형 질의 (gold가 있는 것) |
| 검색 (4단계) | **MRR@k** | 첫 정답 근거 순위의 역수 평균 | 동일 |
| 생성 (6·7단계) | 필수 요소 포함률, 확답 회피, 금지 표현, 근거 충실도 | 5절 루브릭 | 핵심 12문항 (+ 변형 표본) |
| 플로우 (7단계) | 범위 밖 처리 정확도, 오탐률 | 범위 밖 질문을 거르면서 정상 질문을 막지 않는가 | 전체 |

정답 근거 매칭 규칙: 검색 결과 청크의 메타데이터가 **(법령명, 조문번호)** 또는 **사건번호**와 일치하면 정답으로 본다.
따라서 3단계 메타데이터 스키마는 이 두 키를 **반드시** 가져야 한다. (→ 3단계 요구사항)

In [11]:
def recall_at_k(ranked_keys: list[list[str]], gold_keys: list[set[str]], k: int) -> float:
  hits = [bool(set(r[:k]) & g) for r, g in zip(ranked_keys, gold_keys) if g]
  return sum(hits) / len(hits) if hits else 0.0


def mrr_at_k(ranked_keys: list[list[str]], gold_keys: list[set[str]], k: int) -> float:
  scores = []
  for r, g in zip(ranked_keys, gold_keys):
    if not g:
      continue
    rank = next((i + 1 for i, key in enumerate(r[:k]) if key in g), None)
    scores.append(1 / rank if rank else 0.0)
  return sum(scores) / len(scores) if scores else 0.0


# 동작 확인용 예시
_ranked = [["민법|제636조", "민법|제635조"], ["판례|2022다247187"], ["민법|제1조"]]
_gold = [{"민법|제635조"}, {"판례|2022다247187"}, {"민법|제618조"}]
print("Recall@1", recall_at_k(_ranked, _gold, 1), "Recall@2", recall_at_k(_ranked, _gold, 2))
print("MRR@2", round(mrr_at_k(_ranked, _gold, 2), 3))

Recall@1 0.3333333333333333 Recall@2 0.6666666666666666
MRR@2 0.5


## 7. 결과 리포트

> 아래는 이 노트북을 실행한 결과를 바탕으로 작성했다. 평가셋을 수정하면 다시 실행해 갱신한다.

### 7-1. 정답 근거 검증 결과

| 구분 | 가설 | 확인 | 비고 |
|---|---|---|---|
| 조문 | 17건 (민법 12, 공동주택관리법·형법·부동산거래신고법 등 5) | **17건 모두 존재·제목 일치** | 번호만 적었던 하위 법령 2건은 조문을 특정함 (아래) |
| 판례 | Q9 "가계약금" | 대법원 2022다247187 (임차보증가계약금반환) | 참조조문이 가설(제565조)과 다름 → 평가셋 수정 |
| 법령용어 | Q6 "차임" | 정확히 일치하는 항목은 **법령한영사전 1건("rent")** | 사회초년생 설명용 정의로는 쓸 수 없음 |

**하위 법령 조문 특정**
- Q12 시행령: `부동산 거래신고 등에 관한 법률 시행령` **제4조의3**(주택 임대차 계약의 신고)
- Q7 규칙: `공동주택 층간소음의 범위와 기준에 관한 규칙` **제2조**(범위), **제3조**(기준)

### 7-2. 발견과 다음 단계에 주는 영향

1. **판례 판결요지가 비어 있을 수 있다.** Q9 정답 판례는 `판결요지`가 빈 문자열이고, 핵심이 `판시사항`에 있다.
   기존 청커는 `판결요지 + 【이유】 이후 본문`만 임베딩했으므로 이 판례의 핵심 문장이 검색 대상에서 빠졌을 가능성이 크다.
   → **3단계**: 판례 스키마에 `판시사항`·`참조조문`을 필수 필드로 넣고, 임베딩 텍스트 구성을 실험 변수로 둔다.
2. **판례의 참조조문은 조문-판례 연결 정보다.** 2022다247187은 민법 제105조·제568조를 참조한다.
   → **3단계**: `참조조문`을 파싱해 (법령명, 조문번호) 키로 저장하면, 조문을 찾았을 때 관련 판례를 함께 가져오는 확장 검색이 가능하다.
3. **법령용어 API만으로는 용어 질문(Q6)에 답할 수 없다.** 쉬운 정의는 민법 제618조(임대차의 의의)나 일상용어 연계 API에서 가져와야 한다.
   → **2단계**: 법령용어-일상용어 연계 API의 실제 응답을 확인하고, 용어 질문의 근거 소스를 정한다.
4. **법령마다 시행일이 다르다.** 민법 2026-03-17, 공동주택관리법 2026-07-01, 형법 2026-09-13 등 최근 개정이 잦다.
   → **3단계**: 청크 메타데이터에 `시행일자`와 수집일을 넣고, 재수집 시 변경된 조문만 다시 임베딩하도록 매니페스트를 설계한다.
5. **평가 키 형식이 정해졌다.** 조문은 `법령명|제N조(의M)`, 판례는 `판례|사건번호`로 매칭한다.
   → **3단계** 스키마는 이 키를 만들 수 있는 필드(법령명, 조문번호, 조문가지번호, 사건번호)를 반드시 가져야 한다.

### 7-3. 시점별 법령과 근거 링크 (v0.3 추가)

법령은 시행일마다 내용이 달라서, "그때 계약했으면 어떤 법이 적용돼?"라는 질문에는 **그 시점의 조문 판본**이 필요하다.
법령 API의 시행일 법령 목록(`target=eflaw`)으로 판본별 시행일·공포번호를 받을 수 있음을 확인했다.

| 조문 | 확인 결과 | 평가 항목 |
|---|---|---|
| 주택임대차보호법 제6조의3(계약갱신 요구 등) | 2018.10.16 판본에는 **없고**, 2020.7.31 판본부터 있음 | T01 (as_of 2020-10-01) |
| 부동산 거래신고법 제6조의2(주택 임대차 계약의 신고) | 2020.8.18 공포, **2021.6.1 시행** | T02 (as_of 2021-05-15, 시행 전 시점) |

- 시점 질문의 정답은 "그 시점에 시행 중인 판본"이고, 아직 시행 전이면 "가장 먼저 시행되는 판본"이다. 이 규칙은 하네스(`harness/prepare.py`의 `gold_keys`)에 고정했다.
- **근거 링크**: 답변에 인용한 근거마다 law.go.kr 퍼머링크를 준다. 과거 판본 조문은 `/법령/{법령명}/({공포번호},{공포일자})/제N조` 형식으로 열린다.
  law.go.kr은 없는 페이지도 HTTP 200 '오류페이지'로 응답하므로, 링크 검증은 **페이지 제목**으로 해야 한다.
- API 응답의 상세링크 필드(`/DRF/lawService.do?...`)에는 **인증키가 그대로 들어 있다.** 원본 응답을 저장하거나 사용자에게 링크를 보여줄 때는 퍼머링크만 쓴다.
- 기존 서비스의 법령 인용 링크는 법령ID를 `lsiSeq`로 넣어 **다른 법령이 열리는** 오류가 있었다(예: 민법 → 사방사업법). 별도 PR(`fix/oc-leak`)에서 퍼머링크로 고쳤다.

### 7-4. 한계

- 평가셋이 45개 질의(핵심 14 + 변형 31)라서 검색 지표의 신뢰구간이 넓다. 차이가 작으면(대략 1~2문항) 결론을 내리지 않는다.
- 변형 질문은 작성자 1명이 만들어 말투 다양성이 부족할 수 있다. 팀원 검토 후 v0.3에서 보강한다.
- `must_include`는 법률 전문가 검토를 거치지 않았다. 서비스 공개 전에 검토가 필요하다.
- 규칙 기반 확답 필터는 1차 거름망일 뿐이며, 최종 판정은 6단계 LLM 평가자와 사람 채점으로 한다.